# Auditing Before Automation: Shortcut-Aware CXR Triage

This complete, Colab-ready notebook reproduces the computational workflow supporting the manuscript *Auditing Before Automation: Shortcut-Aware Hierarchical Selective Triage of Pneumonia and Tuberculosis on Chest Radiographs*. No convolutional network is trained over repeated epochs. A pretrained MobileNetV3 encoder is frozen and used once to extract image representations; lightweight classifiers are then trained on those representations.

The workflow compares:

- **Direct classification:** Normal versus Pneumonia versus Tuberculosis using Logistic Regression and Extra Trees.
- **Hierarchical classification:** Normal versus abnormal, followed by Pneumonia versus Tuberculosis.

The downloaded dataset is audited before modelling because its supplied partitions contain extensive exact duplication and contradictory labels. Exact conflicts are excluded, one representative per consistent identity is retained, perceptual near-duplicates are grouped, and new group-aware train/validation/calibration/test partitions are created. The notebook then performs model selection, calibration, paired bootstrap analysis, conformal prediction, selective referral, shortcut testing, figure generation, and a complete artifact export.

> **Evidence boundary:** The reconstructed test set is internal. Patient identifiers and original acquisition-source identifiers are unavailable; therefore, patient independence, source independence, and external clinical validity cannot be claimed.


## 1. Environment, automatic dataset retrieval, and configuration

The dataset is stored on the Colab runtime disk. If a reset has removed it, the public Kaggle dataset is downloaded automatically. Outputs are written locally and compressed for browser download at the end. The only GPU-intensive operation is one frozen feature-extraction pass.


In [ ]:
import importlib.util, subprocess, sys, os, json, random, hashlib, warnings, gc, shutil, zipfile
from pathlib import Path

REQUIRED = {"timm": "timm>=1.0.19", "imagehash": "ImageHash>=4.3.2", "seaborn": "seaborn>=0.13.2"}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import imagehash
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
from PIL import Image, ImageEnhance, ImageFilter
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.calibration import calibration_curve
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             log_loss, precision_recall_fscore_support, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

warnings.filterwarnings("ignore", category=UserWarning)

CONFIG = {
    "seed": 20260815,
    "dataset_root": "/content/kaggle_data",
    "dataset_slug": "muhammadrehan00/chest-xray-dataset",
    "output_root": "/content/Auditing_Before_Automation_CXR_Triage",
    "image_size": 160,
    "batch_size": 128,
    "num_workers": 4,
    "encoder": "mobilenetv3_small_100",
    "near_duplicate_hamming": 4,
    "bootstrap_replicates": 500,
    "conformal_alpha": 0.10,
    "target_auto_coverage": 0.75,
    "target_abnormal_sensitivity": 0.98,
    "target_tb_sensitivity": 0.95,
    "run_optional_robustness": False,
    "auto_download_results": True,
}

SEED = CONFIG["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

REQUESTED_ROOT = Path(CONFIG["dataset_root"])
def valid_root(root):
    return all((root / split / label).is_dir() for split in ("train", "val", "test")
               for label in ("normal", "pneumonia", "tuberculosis"))

def locate_root(base):
    candidates = [base]
    if base.exists(): candidates += [path.parent for path in base.rglob("train") if path.is_dir()]
    for candidate in candidates:
        if valid_root(candidate): return candidate.resolve()
    return None

DATA_ROOT = locate_root(REQUESTED_ROOT)
if DATA_ROOT is None:
    print("Dataset absent; downloading from Kaggle...")
    if importlib.util.find_spec("kagglehub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub>=0.3.12"])
    import kagglehub
    downloaded = kagglehub.dataset_download(CONFIG["dataset_slug"], output_dir=str(REQUESTED_ROOT))
    DATA_ROOT = locate_root(REQUESTED_ROOT) or locate_root(Path(downloaded))
if DATA_ROOT is None: raise FileNotFoundError("The expected three-class train/val/test structure was not found.")

OUTPUT_DIR = Path(CONFIG["output_root"])
FIG_DIR, TAB_DIR, RAW_DIR, MODEL_DIR = [OUTPUT_DIR / name for name in ("figures", "tables", "raw", "models")]
for directory in (OUTPUT_DIR, FIG_DIR, TAB_DIR, RAW_DIR, MODEL_DIR): directory.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "outputs_summary.txt"
def log(message=""):
    print(message, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as stream: stream.write(str(message) + "\n")

log(json.dumps({"device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                "dataset_root": str(DATA_ROOT), "config": CONFIG}, indent=2))


## 2. Image audit, contradictory-label removal, and group-aware partitions

Every image is checked for readability and assigned SHA-256 and perceptual hashes. Byte-identical identities carrying contradictory labels are excluded completely. One representative is retained from each consistent SHA-256 identity. Perceptually similar representatives are grouped, and cross-label perceptual groups are also excluded. Approximately 70%, 10%, 10%, and 10% are assigned to training, validation, calibration, and internal testing, respectively.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

CLASS_NAMES = ["Normal", "Pneumonia", "Tuberculosis"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
FOLDER_TO_CLASS = {name.lower(): name for name in CLASS_NAMES}

def audit_one(record):
    result = dict(record); path = Path(record["path"])
    result.update({"readable": False, "sha256": "", "phash": "", "width": np.nan, "height": np.nan,
                   "mean": np.nan, "std": np.nan, "sharpness": np.nan, "dark_fraction": np.nan,
                   "bright_fraction": np.nan, "error": ""})
    try:
        raw = path.read_bytes()
        with Image.open(path) as image:
            image.load(); gray = image.convert("L"); width, height = image.size
            thumb = gray.copy(); thumb.thumbnail((192, 192), Image.Resampling.BILINEAR)
            array = np.asarray(thumb, dtype=np.float32) / 255.0
            lap = -4*array + np.roll(array,1,0)+np.roll(array,-1,0)+np.roll(array,1,1)+np.roll(array,-1,1)
            result.update({"readable": True, "sha256": hashlib.sha256(raw).hexdigest(),
                "phash": str(imagehash.phash(gray)), "width": width, "height": height,
                "mean": float(array.mean()), "std": float(array.std()), "sharpness": float(lap.var()),
                "dark_fraction": float((array < .02).mean()), "bright_fraction": float((array > .98).mean())})
    except Exception as exc: result["error"] = repr(exc)
    return result

records = []
for source_split in ("train", "val", "test"):
    for folder, class_name in FOLDER_TO_CLASS.items():
        for path in sorted((DATA_ROOT/source_split/folder).glob("*.jpg")):
            records.append({"path": str(path), "source_split": source_split, "class_name": class_name})
inventory = pd.DataFrame(records)
audit_file = TAB_DIR / "image_audit.csv"
if audit_file.exists():
    audit_df = pd.read_csv(audit_file); log(f"Loaded image audit: {len(audit_df):,}")
else:
    completed = []
    with ThreadPoolExecutor(max_workers=4) as pool:
        for index, result in enumerate(pool.map(audit_one, inventory.to_dict("records")), 1):
            completed.append(result)
            if index % 500 == 0 or index == len(inventory): log(f"Audited {index:,}/{len(inventory):,} images")
    audit_df = pd.DataFrame(completed); audit_df.to_csv(audit_file, index=False)

audit_df["readable"] = audit_df["readable"].astype(bool)
valid = audit_df.loc[audit_df.readable].copy().reset_index(drop=True)
exact_summary = valid.groupby("sha256").agg(n=("path","size"), n_classes=("class_name","nunique")).reset_index()
conflict_hashes = set(exact_summary.loc[exact_summary.n_classes > 1, "sha256"])
exact_conflicts = valid.loc[valid.sha256.isin(conflict_hashes)].copy()
exact_conflicts.to_csv(TAB_DIR / "excluded_exact_label_conflicts.csv", index=False)
exact_conflict_pairs=(exact_conflicts.groupby("sha256").class_name
                      .apply(lambda values: "|".join(sorted(set(values))))
                      .value_counts().rename_axis("class_pair").rename("identity_count").reset_index())
exact_conflict_pairs.to_csv(TAB_DIR / "exact_conflict_class_pairs.csv", index=False)
unique_df = (valid.loc[~valid.sha256.isin(conflict_hashes)].sort_values(["sha256","path"])
             .drop_duplicates("sha256").reset_index(drop=True))

class UnionFind:
    def __init__(self, n): self.parent=list(range(n)); self.rank=[0]*n
    def find(self,x):
        while self.parent[x]!=x: self.parent[x]=self.parent[self.parent[x]]; x=self.parent[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1

class BKNode:
    def __init__(self,value,index): self.value=value; self.indices=[index]; self.children={}
class BKTree:
    def __init__(self): self.root=None
    def add(self,value,index):
        if self.root is None:self.root=BKNode(value,index);return
        node=self.root
        while True:
            distance=value-node.value
            if distance==0:node.indices.append(index);return
            if distance not in node.children:node.children[distance]=BKNode(value,index);return
            node=node.children[distance]
    def query(self,value,radius):
        if self.root is None:return []
        found=[]; stack=[self.root]
        while stack:
            node=stack.pop(); distance=value-node.value
            if distance<=radius:found.extend(node.indices)
            stack.extend(child for edge,child in node.children.items() if distance-radius<=edge<=distance+radius)
        return found

uf=UnionFind(len(unique_df)); tree=BKTree()
for index,text_hash in enumerate(unique_df.phash):
    value=imagehash.hex_to_hash(text_hash)
    for neighbor in tree.query(value, CONFIG["near_duplicate_hamming"]): uf.union(index,neighbor)
    tree.add(value,index)
roots=[uf.find(index) for index in range(len(unique_df))]
mapping={root:number for number,root in enumerate(sorted(set(roots)))}
unique_df["duplicate_group"]=[mapping[root] for root in roots]
component_summary=unique_df.groupby("duplicate_group").agg(n=("path","size"),n_classes=("class_name","nunique")).reset_index()
near_conflict_groups=set(component_summary.loc[component_summary.n_classes>1,"duplicate_group"])
near_conflicts=unique_df.loc[unique_df.duplicate_group.isin(near_conflict_groups)].copy()
near_conflicts.to_csv(TAB_DIR / "excluded_near_duplicate_label_conflicts.csv", index=False)
clean_df=unique_df.loc[~unique_df.duplicate_group.isin(near_conflict_groups)].copy().reset_index(drop=True)
clean_df["label"]=clean_df.class_name.map(CLASS_TO_INDEX).astype(int)
exclusion_class_counts=pd.concat([
    exact_conflicts.assign(exclusion_type="exact_label_conflict"),
    near_conflicts.assign(exclusion_type="near_duplicate_label_conflict")
],ignore_index=True).groupby(["exclusion_type","class_name"]).size().rename("n_files").reset_index()
exclusion_class_counts.to_csv(TAB_DIR / "excluded_conflict_class_counts.csv", index=False)
retained_signature_summary=(clean_df.groupby("class_name")[["width","height","mean","std","sharpness","dark_fraction","bright_fraction"]]
                            .agg(["count","median","mean"]))
retained_signature_summary.to_csv(TAB_DIR / "retained_acquisition_signature_summary.csv")

def holdout(frame,n_splits,seed):
    splitter=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    keep,take=next(splitter.split(frame,frame.label,frame.duplicate_group))
    return frame.iloc[keep].reset_index(drop=True),frame.iloc[take].reset_index(drop=True)
remaining,test_df=holdout(clean_df,10,SEED)
remaining,calibration_df=holdout(remaining,9,SEED+1)
train_df,val_df=holdout(remaining,8,SEED+2)
frames={"train":train_df,"validation":val_df,"calibration":calibration_df,"test":test_df}
assignment=pd.concat([frame.assign(experimental_split=name) for name,frame in frames.items()],ignore_index=True)
assert assignment.groupby("sha256").experimental_split.nunique().max()==1
assert assignment.groupby("duplicate_group").experimental_split.nunique().max()==1
for name,frame in frames.items(): frame.to_csv(TAB_DIR/f"split_{name}.csv",index=False)

integrity={"source_files":len(audit_df),"unique_sha256":int(valid.sha256.nunique()),
           "exact_conflict_identities":len(conflict_hashes),"exact_conflict_files":len(exact_conflicts),
           "near_conflict_groups":len(near_conflict_groups),"near_conflict_images":len(near_conflicts),
           "final_clean_images":len(clean_df),"class_counts":clean_df.class_name.value_counts().to_dict()}
(RAW_DIR/"integrity_audit.json").write_text(json.dumps(integrity,indent=2),encoding="utf-8")
split_counts=assignment.groupby(["experimental_split","class_name"]).size().rename("n").reset_index()
split_counts.to_csv(TAB_DIR/"split_counts.csv",index=False)
log("Integrity audit:\n"+json.dumps(integrity,indent=2)); log("Splits:\n"+split_counts.to_string(index=False))


## 3. One-pass frozen MobileNetV3 feature extraction

Images are resized to 160×160 and passed through a pretrained MobileNetV3-Small encoder with all parameters frozen. No epoch-based neural-network training is performed. Embeddings are cached, so rerunning subsequent sections does not repeat image decoding or GPU inference.


In [ ]:
normalization_mean=[0.485,0.456,0.406]; normalization_std=[0.229,0.224,0.225]
transform=transforms.Compose([transforms.Resize((CONFIG["image_size"],CONFIG["image_size"]),interpolation=InterpolationMode.BILINEAR),
                              transforms.ToTensor(),transforms.Normalize(normalization_mean,normalization_std)])
class ImageDataset(Dataset):
    def __init__(self,frame,transform):
        self.paths=frame.path.tolist(); self.labels=frame.label.to_numpy(np.int64); self.transform=transform
    def __len__(self):return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as image: tensor=self.transform(image.convert("RGB"))
        return tensor,int(self.labels[index])

def make_loader(frame):
    return DataLoader(ImageDataset(frame,transform),batch_size=CONFIG["batch_size"],shuffle=False,
                      num_workers=CONFIG["num_workers"],pin_memory=DEVICE.type=="cuda",
                      persistent_workers=CONFIG["num_workers"]>0)

embedding_file=RAW_DIR/"frozen_embeddings.npz"
if embedding_file.exists():
    cache=np.load(embedding_file)
    embeddings={name:cache[f"x_{name}"] for name in frames}; labels={name:cache[f"y_{name}"] for name in frames}
    log("Loaded cached frozen embeddings.")
else:
    encoder=timm.create_model(CONFIG["encoder"],pretrained=True,num_classes=0,global_pool="avg").to(DEVICE).eval()
    for parameter in encoder.parameters():parameter.requires_grad=False
    embeddings={};labels={}
    with torch.inference_mode():
        for name,frame in frames.items():
            feature_batches=[];label_batches=[]
            for batch_number,(images,target) in enumerate(make_loader(frame),1):
                with torch.amp.autocast(device_type=DEVICE.type,enabled=AMP_ENABLED): features=encoder(images.to(DEVICE,non_blocking=True))
                feature_batches.append(features.float().cpu().numpy());label_batches.append(target.numpy())
                if batch_number%20==0:log(f"{name}: extracted {min(batch_number*CONFIG['batch_size'],len(frame)):,}/{len(frame):,}")
            embeddings[name]=np.concatenate(feature_batches);labels[name]=np.concatenate(label_batches)
            log(f"{name}: embedding matrix {embeddings[name].shape}")
    np.savez_compressed(embedding_file,**{f"x_{name}":value for name,value in embeddings.items()},
                        **{f"y_{name}":value for name,value in labels.items()})
    encoder.cpu();del encoder;gc.collect();torch.cuda.empty_cache()

X_train,X_val,X_cal,X_test=[embeddings[name] for name in ("train","validation","calibration","test")]
y_train,y_val,y_cal,y_test=[labels[name] for name in ("train","validation","calibration","test")]


## 4. Fast direct and hierarchical classifiers

The direct benchmark compares a balanced multinomial Logistic Regression model with a nonlinear Extra Trees model. Validation macro-F1 selects the direct model. The hierarchy uses two balanced logistic heads: Normal versus abnormal and Pneumonia versus Tuberculosis. All models operate on the same frozen representations.


In [ ]:
direct_candidates={}
for c_value in (0.1,1.0,10.0):
    model=make_pipeline(StandardScaler(),LogisticRegression(C=c_value,class_weight="balanced",max_iter=400,solver="lbfgs"))
    model.fit(X_train,y_train); score=f1_score(y_val,model.predict(X_val),average="macro")
    direct_candidates[f"Logistic_C{c_value}"]=(model,score);log(f"Logistic C={c_value}: validation macro-F1={score:.5f}")
forest=ExtraTreesClassifier(n_estimators=250,min_samples_leaf=2,class_weight="balanced",n_jobs=-1,random_state=SEED)
forest.fit(X_train,y_train);forest_score=f1_score(y_val,forest.predict(X_val),average="macro")
direct_candidates["ExtraTrees"]=(forest,forest_score);log(f"Extra Trees: validation macro-F1={forest_score:.5f}")
best_direct_name=max(direct_candidates,key=lambda name:direct_candidates[name][1])
best_direct_model=direct_candidates[best_direct_name][0]
validation_comparison=pd.DataFrame([{"model":name,"validation_macro_f1":value[1]} for name,value in direct_candidates.items()])
validation_comparison.to_csv(TAB_DIR/"validation_model_comparison.csv",index=False)
joblib.dump(best_direct_model,MODEL_DIR/"selected_direct_model.joblib")
log(f"Selected direct model: {best_direct_name}")

abnormal_train=(y_train!=CLASS_TO_INDEX["Normal"]).astype(int)
abnormal_val=(y_val!=CLASS_TO_INDEX["Normal"]).astype(int)
abnormal_model=make_pipeline(StandardScaler(),LogisticRegression(C=1.0,class_weight="balanced",max_iter=400))
abnormal_model.fit(X_train,abnormal_train)
disease_train_mask=y_train!=CLASS_TO_INDEX["Normal"]
disease_val_mask=y_val!=CLASS_TO_INDEX["Normal"]
disease_train=(y_train[disease_train_mask]==CLASS_TO_INDEX["Tuberculosis"]).astype(int)
disease_val=(y_val[disease_val_mask]==CLASS_TO_INDEX["Tuberculosis"]).astype(int)
disease_model=make_pipeline(StandardScaler(),LogisticRegression(C=1.0,class_weight="balanced",max_iter=400))
disease_model.fit(X_train[disease_train_mask],disease_train)
joblib.dump(abnormal_model,MODEL_DIR/"hierarchy_abnormal.joblib");joblib.dump(disease_model,MODEL_DIR/"hierarchy_disease.joblib")
log(f"Hierarchy validation macro-F1: abnormal={f1_score(abnormal_val,abnormal_model.predict(X_val),average='macro'):.5f}; disease={f1_score(disease_val,disease_model.predict(X_val[disease_val_mask]),average='macro'):.5f}")


## 5. Validation-only temperature calibration and probability construction

Temperature parameters are fitted only on validation data. A separate calibration partition remains reserved for conformal and operating thresholds. Hierarchical probabilities follow \(p(N)=1-p(A)\), \(p(P)=p(A)p(P\mid A)\), and \(p(T)=p(A)p(T\mid A)\).


In [ ]:
def logits_for(model,x):
    if hasattr(model,"decision_function"):
        scores=model.decision_function(x)
        return np.column_stack([-scores/2,scores/2]) if scores.ndim==1 else scores
    probabilities=np.clip(model.predict_proba(x),1e-8,1);return np.log(probabilities)
def fit_temperature(logits,target):
    result=minimize_scalar(lambda log_t:log_loss(target,softmax(logits/np.exp(log_t),axis=1),labels=np.arange(logits.shape[1])),
                           bounds=(-3,3),method="bounded")
    return float(np.exp(result.x))
def calibrated_probs(model,x,temperature):return softmax(logits_for(model,x)/temperature,axis=1)

direct_temperature=fit_temperature(logits_for(best_direct_model,X_val),y_val)
direct_val_probs=calibrated_probs(best_direct_model,X_val,direct_temperature)
direct_cal_probs=calibrated_probs(best_direct_model,X_cal,direct_temperature)
direct_test_probs=calibrated_probs(best_direct_model,X_test,direct_temperature)
abnormal_temperature=fit_temperature(logits_for(abnormal_model,X_val),abnormal_val)
disease_temperature=fit_temperature(logits_for(disease_model,X_val[disease_val_mask]),disease_val)
def hierarchy_probs(x):
    p_abnormal=calibrated_probs(abnormal_model,x,abnormal_temperature)[:,1]
    conditional=calibrated_probs(disease_model,x,disease_temperature)
    return np.column_stack([1-p_abnormal,p_abnormal*conditional[:,0],p_abnormal*conditional[:,1]])
hierarchical_val_probs=hierarchy_probs(X_val);hierarchical_cal_probs=hierarchy_probs(X_cal);hierarchical_test_probs=hierarchy_probs(X_test)
temperatures={"direct":direct_temperature,"abnormal":abnormal_temperature,"disease":disease_temperature}
(RAW_DIR/"temperatures.json").write_text(json.dumps(temperatures,indent=2),encoding="utf-8")
np.savez_compressed(RAW_DIR/"analysis_state.npz",y_val=y_val,y_cal=y_cal,y_test=y_test,
                    direct_val_probs=direct_val_probs,direct_cal_probs=direct_cal_probs,direct_test_probs=direct_test_probs,
                    hierarchical_val_probs=hierarchical_val_probs,hierarchical_cal_probs=hierarchical_cal_probs,
                    hierarchical_test_probs=hierarchical_test_probs)
log("Temperatures: "+json.dumps(temperatures))


## 6. Internal test metrics and bootstrap confidence intervals

The direct and hierarchical models are evaluated using accuracy, balanced accuracy, macro-F1, weighted-F1, macro one-versus-rest AUC, negative log-likelihood, Brier score, expected calibration error, and class-specific precision, sensitivity, specificity, and F1. Five hundred class-stratified bootstrap replicates are used by default and progress is printed every 100 replicates.


In [ ]:
def brier(y,p):return float(np.mean(np.sum((p-np.eye(3)[y])**2,axis=1)))
def ece(y,p,bins=15):
    pred=p.argmax(1);conf=p.max(1);correct=pred==y;edges=np.linspace(0,1,bins+1);value=0
    for index in range(bins):
        mask=(conf>=edges[index])&(conf<edges[index+1] if index<bins-1 else conf<=edges[index+1])
        if mask.any():value+=mask.mean()*abs(correct[mask].mean()-conf[mask].mean())
    return float(value)
def metrics(y,p):
    pred=p.argmax(1);matrix=confusion_matrix(y,pred,labels=np.arange(3));precision,recall,class_f1,support=precision_recall_fscore_support(y,pred,labels=np.arange(3),zero_division=0)
    result={"accuracy":accuracy_score(y,pred),"balanced_accuracy":balanced_accuracy_score(y,pred),
            "macro_f1":f1_score(y,pred,average="macro"),"weighted_f1":f1_score(y,pred,average="weighted"),
            "macro_auc_ovr":roc_auc_score(y,p,multi_class="ovr",average="macro"),"ece":ece(y,p),"brier":brier(y,p),
            "nll":log_loss(y,p,labels=np.arange(3))}
    for k,name in enumerate(CLASS_NAMES):
        tp=matrix[k,k];fn=matrix[k].sum()-tp;fp=matrix[:,k].sum()-tp;tn=matrix.sum()-tp-fn-fp
        result.update({f"{name}_precision":precision[k],f"{name}_sensitivity":recall[k],f"{name}_specificity":tn/max(tn+fp,1),f"{name}_f1":class_f1[k],f"{name}_support":int(support[k])})
    return result

probabilities={"direct":direct_test_probs,"hierarchical":hierarchical_test_probs}
metrics_df=pd.DataFrame([{"model":name,**metrics(y_test,p)} for name,p in probabilities.items()])
metrics_df.to_csv(TAB_DIR/"test_metrics.csv",index=False);log("Test metrics:\n"+metrics_df.round(5).to_string(index=False))
bootstrap_rows=[];groups=[np.flatnonzero(y_test==k) for k in range(3)]
for replicate in range(CONFIG["bootstrap_replicates"]):
    rng=np.random.default_rng(SEED+replicate);indices=np.concatenate([rng.choice(group,len(group),replace=True) for group in groups])
    for name,p in probabilities.items():bootstrap_rows.append({"replicate":replicate,"model":name,**metrics(y_test[indices],p[indices])})
    if (replicate+1)%100==0:log(f"Bootstrap {replicate+1}/{CONFIG['bootstrap_replicates']}")
bootstrap_df=pd.DataFrame(bootstrap_rows);bootstrap_df.to_csv(TAB_DIR/"bootstrap_samples.csv",index=False)
ci_rows=[]
for name,group in bootstrap_df.groupby("model"):
    for metric in ("accuracy","balanced_accuracy","macro_f1","macro_auc_ovr","ece","brier","nll"):
        ci_rows.append({"model":name,"metric":metric,"ci_low":group[metric].quantile(.025),"ci_high":group[metric].quantile(.975)})
pd.DataFrame(ci_rows).to_csv(TAB_DIR/"bootstrap_95ci.csv",index=False)


## 7. Conformal prediction and calibration-locked selective triage

Class-conditional conformal thresholds and sensitivity-aware operating thresholds are locked using the calibration partition. Automatic decisions require agreement between the hierarchical decision and a conformal singleton, together with normalized predictive entropy below the calibration-defined 75% quantile.


In [ ]:
def higher_quantile(values,alpha):
    level=min(1.0,np.ceil((len(values)+1)*(1-alpha))/len(values));return float(np.quantile(values,level,method="higher"))
thresholds={k:higher_quantile(1-hierarchical_cal_probs[y_cal==k,k],CONFIG["conformal_alpha"]) for k in range(3)}
sets=[[k for k in range(3) if 1-row[k]<=thresholds[k]] for row in hierarchical_test_probs]
covered=np.array([target in prediction_set for target,prediction_set in zip(y_test,sets)]);sizes=np.array(list(map(len,sets)))
conformal={"marginal_coverage":covered.mean(),"mean_set_size":sizes.mean(),"singleton_rate":np.mean(sizes==1),
           **{f"{name}_coverage":covered[y_test==k].mean() for k,name in enumerate(CLASS_NAMES)}}
pd.DataFrame([conformal]).to_csv(TAB_DIR/"conformal_summary.csv",index=False)

def sensitivity_threshold(y_binary,score,target):
    _,tpr,values=roc_curve(y_binary,score);feasible=np.flatnonzero(tpr>=target);return float(values[feasible[0]])
cal_abnormal=(y_cal!=0).astype(int);abnormal_score=1-hierarchical_cal_probs[:,0]
abnormal_threshold=sensitivity_threshold(cal_abnormal,abnormal_score,CONFIG["target_abnormal_sensitivity"])
mask=y_cal!=0;conditional=hierarchical_cal_probs[mask,1:];conditional/=conditional.sum(1,keepdims=True)
tb_threshold=sensitivity_threshold((y_cal[mask]==2).astype(int),conditional[:,1],CONFIG["target_tb_sensitivity"])
def entropy(p):
    p=np.clip(p,1e-8,1);return -np.sum(p*np.log(p),axis=1)/np.log(3)
entropy_threshold=float(np.quantile(entropy(hierarchical_cal_probs),CONFIG["target_auto_coverage"]))
def operating_prediction(p):
    prediction=np.zeros(len(p),dtype=int);abnormal=(1-p[:,0])>=abnormal_threshold;conditional=p[:,1:]/p[:,1:].sum(1,keepdims=True)
    prediction[abnormal]=np.where(conditional[abnormal,1]>=tb_threshold,2,1);return prediction
operating_pred=operating_prediction(hierarchical_test_probs);singleton=np.array([len(value)==1 for value in sets]);single_label=np.array([value[0] if len(value)==1 else -1 for value in sets])
accepted=singleton&(single_label==operating_pred)&(entropy(hierarchical_test_probs)<=entropy_threshold)
triage={"coverage":accepted.mean(),"accepted_accuracy":accuracy_score(y_test[accepted],operating_pred[accepted]) if accepted.any() else np.nan,
        "selective_risk":1-accuracy_score(y_test[accepted],operating_pred[accepted]) if accepted.any() else np.nan,"referred_n":int((~accepted).sum())}
pd.DataFrame([triage]).to_csv(TAB_DIR/"selective_triage.csv",index=False)
pd.DataFrame([{"abnormal_threshold":abnormal_threshold,"tb_threshold":tb_threshold,
               "entropy_threshold":entropy_threshold,"entropy_quantile":CONFIG["target_auto_coverage"]}]).to_csv(TAB_DIR/"operating_thresholds.csv",index=False)
referral_composition=(pd.DataFrame({"true_class":[CLASS_NAMES[k] for k in y_test],"accepted":accepted})
                      .groupby(["true_class","accepted"]).size().rename("n").reset_index())
referral_composition["decision"]=np.where(referral_composition.accepted,"accepted","referred")
referral_composition.to_csv(TAB_DIR/"referral_composition_by_class.csv",index=False)
if CONFIG["run_optional_robustness"]:
    sensitivity_rows=[]
    for entropy_quantile in (0.50,0.75,0.90):
        alt_entropy_threshold=float(np.quantile(entropy(hierarchical_cal_probs),entropy_quantile))
        alt_accepted=singleton&(single_label==operating_pred)&(entropy(hierarchical_test_probs)<=alt_entropy_threshold)
        alt_accuracy=accuracy_score(y_test[alt_accepted],operating_pred[alt_accepted]) if alt_accepted.any() else np.nan
        sensitivity_rows.append({"entropy_quantile":entropy_quantile,"entropy_threshold":alt_entropy_threshold,
                                 "coverage":alt_accepted.mean(),"accepted_accuracy":alt_accuracy,
                                 "selective_risk":1-alt_accuracy if np.isfinite(alt_accuracy) else np.nan,
                                 "referred_n":int((~alt_accepted).sum())})
    pd.DataFrame(sensitivity_rows).to_csv(TAB_DIR/"entropy_threshold_sensitivity.csv",index=False)
curve=[];confidence=hierarchical_test_probs.max(1)
for coverage in np.linspace(.1,1,19):
    count=max(1,round(coverage*len(y_test)));chosen=np.argsort(-confidence)[:count];curve.append({"coverage":count/len(y_test),"risk":1-accuracy_score(y_test[chosen],hierarchical_test_probs.argmax(1)[chosen])})
coverage_risk_df=pd.DataFrame(curve);coverage_risk_df.to_csv(TAB_DIR/"coverage_risk.csv",index=False)
log("Conformal: "+json.dumps(conformal));log("Selective triage: "+json.dumps(triage))


## 8. Dataset-shortcut diagnostic

Near-perfect image-model performance can be caused by acquisition or source artifacts. A metadata-only classifier is therefore trained using image dimensions and simple intensity statistics. High metadata-only performance is a warning that the image labels may remain confounded by source characteristics even after duplicate control.


In [ ]:
shortcut_columns=["width","height","mean","std","sharpness","dark_fraction","bright_fraction"]
shortcut_model=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=400))
shortcut_model.fit(train_df[shortcut_columns],train_df.label)
shortcut_pred=shortcut_model.predict(test_df[shortcut_columns])
shortcut_metrics={"accuracy":accuracy_score(y_test,shortcut_pred),"balanced_accuracy":balanced_accuracy_score(y_test,shortcut_pred),
                  "macro_f1":f1_score(y_test,shortcut_pred,average="macro")}
pd.DataFrame([shortcut_metrics]).to_csv(TAB_DIR/"metadata_shortcut_diagnostic.csv",index=False)
log("Metadata-only shortcut diagnostic: "+json.dumps(shortcut_metrics))
if shortcut_metrics["macro_f1"]>=0.70:log("WARNING: strong metadata-only predictability suggests dataset-source confounding.")


## 9. Manuscript-ready figures

Eight figures are generated from the current run: integrity-controlled composition, validation comparison, confusion matrices, ROC curves, reliability, conformal behavior, selective risk, and frozen-representation structure. Numerical values are never pre-populated.


In [ ]:
sns.set_theme(style="whitegrid",context="notebook")
counts=assignment.groupby(["experimental_split","class_name"]).size().rename("n").reset_index()
fig,ax=plt.subplots(figsize=(8,4.5));sns.barplot(data=counts,x="class_name",y="n",hue="experimental_split",ax=ax);ax.set(xlabel="Class",ylabel="Images",title="Leakage-controlled partitions");fig.tight_layout();fig.savefig(FIG_DIR/"fig01_partitions.png",dpi=300);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4));sns.barplot(data=validation_comparison,x="model",y="validation_macro_f1",ax=ax);ax.set_ylim(0,1);ax.tick_params(axis="x",rotation=20);ax.set(title="Direct-model validation comparison",ylabel="Macro-F1",xlabel="Model");fig.tight_layout();fig.savefig(FIG_DIR/"fig02_validation_models.png",dpi=300);plt.show();plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(10,4));
for ax,(name,p) in zip(axes,probabilities.items()):sns.heatmap(confusion_matrix(y_test,p.argmax(1)),annot=True,fmt="d",cmap="Blues",cbar=False,xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,ax=ax);ax.set(title=name.title(),xlabel="Predicted",ylabel="True")
fig.tight_layout();fig.savefig(FIG_DIR/"fig03_confusion.png",dpi=300);plt.show();plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(10,4));
for ax,(name,p) in zip(axes,probabilities.items()):
    for k,class_name in enumerate(CLASS_NAMES):
        fpr,tpr,_=roc_curve(y_test==k,p[:,k]);ax.plot(fpr,tpr,label=f"{class_name}: {roc_auc_score(y_test==k,p[:,k]):.3f}")
    ax.plot([0,1],[0,1],"k--");ax.set(title=name.title(),xlabel="False-positive rate",ylabel="Sensitivity");ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(FIG_DIR/"fig04_roc.png",dpi=300);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(5.5,5));
for name,p in probabilities.items():observed,predicted=calibration_curve((p.argmax(1)==y_test).astype(int),p.max(1),n_bins=10,strategy="quantile");ax.plot(predicted,observed,"o-",label=name)
ax.plot([0,1],[0,1],"k--");ax.set(xlabel="Confidence",ylabel="Observed accuracy",title="Reliability");ax.legend();fig.tight_layout();fig.savefig(FIG_DIR/"fig05_reliability.png",dpi=300);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4));pd.Series(conformal).plot.bar(ax=ax);ax.axhline(1-CONFIG["conformal_alpha"],color="black",linestyle="--");ax.set_ylim(0,max(1.05,conformal["mean_set_size"]+.1));ax.set_title("Conformal behavior");fig.tight_layout();fig.savefig(FIG_DIR/"fig06_conformal.png",dpi=300);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(6,4));ax.plot(coverage_risk_df.coverage,coverage_risk_df.risk,"o-");ax.scatter([triage["coverage"]],[triage["selective_risk"]],color="red",label="Locked policy");ax.set(xlabel="Coverage",ylabel="Risk",title="Coverage-risk relation");ax.legend();fig.tight_layout();fig.savefig(FIG_DIR/"fig07_coverage_risk.png",dpi=300);plt.show();plt.close(fig)
sample=np.random.default_rng(SEED).choice(len(X_test),min(2000,len(X_test)),replace=False);projection=PCA(n_components=2,random_state=SEED).fit_transform(X_test[sample]);plot_df=pd.DataFrame({"PC1":projection[:,0],"PC2":projection[:,1],"Class":[CLASS_NAMES[k] for k in y_test[sample]]});fig,ax=plt.subplots(figsize=(7,5));sns.scatterplot(data=plot_df,x="PC1",y="PC2",hue="Class",s=18,alpha=.65,ax=ax);ax.set_title("Frozen MobileNetV3 representation");fig.tight_layout();fig.savefig(FIG_DIR/"fig08_embedding_pca.png",dpi=300);plt.show();plt.close(fig)


## 10. Exports, audit trail, and downloadable results

All probabilities, predictions, partitions, metrics, confidence intervals, models, figures, and interpretation guardrails are exported. The result archive excludes the 3.05 GB dataset.


In [ ]:
test_export=test_df[["path","source_split","class_name","label","sha256","duplicate_group"]].copy()
for k,name in enumerate(CLASS_NAMES):
    key=name.lower();test_export[f"direct_probability_{key}"]=direct_test_probs[:,k];test_export[f"hierarchical_probability_{key}"]=hierarchical_test_probs[:,k]
test_export["direct_prediction"]=[CLASS_NAMES[k] for k in direct_test_probs.argmax(1)];test_export["hierarchical_prediction"]=[CLASS_NAMES[k] for k in hierarchical_test_probs.argmax(1)]
test_export["conformal_set"]=["|".join(CLASS_NAMES[k] for k in value) if value else "EMPTY" for value in sets];test_export["accepted"] = accepted
test_export.to_csv(TAB_DIR/"test_case_predictions.csv",index=False)
summary=["AUDITING BEFORE AUTOMATION — RUN COMPLETE",f"Dataset root: {DATA_ROOT}",f"Clean images: {len(clean_df):,}",f"Selected direct model: {best_direct_name}","",metrics_df.round(6).to_string(index=False),"",f"Conformal: {conformal}",f"Triage: {triage}",f"Metadata shortcut: {shortcut_metrics}","","GUARDRAILS:","- Internal reconstructed test set; not external validation.","- Patient and acquisition-source independence are unverified.","- High metadata-only performance indicates source confounding."]
LOG_PATH.write_text("\n".join(summary),encoding="utf-8");print("\n".join(summary))
archive_path=Path("/content/Auditing_Before_Automation_CXR_Triage_results.zip")
with zipfile.ZipFile(archive_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():archive.write(path,path.relative_to(OUTPUT_DIR))
print("Results archive:",archive_path)
if CONFIG["auto_download_results"]:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except Exception as exc:print("Download manually from the Files panel:",archive_path,exc)


## Manuscript interpretation checklist

- Report the exact and perceptual conflict exclusions generated by this run.
- Describe MobileNetV3 as a frozen pretrained feature extractor, not as a fine-tuned model.
- Identify validation as the model-selection and temperature-scaling partition.
- Identify calibration as the conformal and operating-threshold partition.
- Describe the test set as reconstructed and internal.
- Report the metadata-only shortcut diagnostic beside predictive performance.
- Do not claim clinical readiness without patient-aware, source-independent external validation.
